# 04 — Speaker ↔ Mouth Track Alignment

ใช้ข้อมูล:
- `dataset/mouth_crops/video001/mouth_metadata.csv`
- `dataset/transcripts/video001/transcript_with_frames.csv`

เป้าหมายคือประเมินว่า transcript แต่ละช่วงน่าจะเป็นของ `track_id` ไหน โดยดู **lip motion** ของ mouth crops

ผลลัพธ์:
- `speaker_track_alignment.csv`
- `speaker_track_alignment_auto.csv`
- `speaker_track_alignment_review.csv`
- `mouth_motion.csv`
- `segment_track_activity.csv`

> ช่วงที่ไม่มั่นใจจะถูกทำเครื่องหมาย `REVIEW` แทนการบังคับ label

## Cell 1 — ตั้งค่า Path

In [1]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

MOUTH_META = PROJECT_ROOT / "dataset" / "mouth_crops" / VIDEO_ID / "mouth_metadata.csv"
TRANSCRIPT_CSV = PROJECT_ROOT / "dataset" / "transcripts" / VIDEO_ID / "transcript_with_frames.csv"

OUTPUT_ROOT = PROJECT_ROOT / "dataset" / "speaker_alignment" / VIDEO_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("MOUTH_META     :", MOUTH_META)
print("TRANSCRIPT_CSV :", TRANSCRIPT_CSV)
print("OUTPUT_ROOT    :", OUTPUT_ROOT)

if not MOUTH_META.exists():
    raise FileNotFoundError(MOUTH_META)

if not TRANSCRIPT_CSV.exists():
    raise FileNotFoundError(TRANSCRIPT_CSV)

MOUTH_META     : D:\Face_detection\dataset\mouth_crops\video001\mouth_metadata.csv
TRANSCRIPT_CSV : D:\Face_detection\dataset\transcripts\video001\transcript_with_frames.csv
OUTPUT_ROOT    : D:\Face_detection\dataset\speaker_alignment\video001


## Cell 2 — โหลดข้อมูล

In [2]:
import pandas as pd

mouth_df = pd.read_csv(MOUTH_META)
transcript_df = pd.read_csv(TRANSCRIPT_CSV)

if mouth_df["accepted"].dtype == object:
    mouth_df["accepted"] = (
        mouth_df["accepted"].astype(str).str.lower().isin(["true", "1", "yes"])
    )

mouth_df = mouth_df[mouth_df["accepted"] == True].copy()
mouth_df["track_id"] = mouth_df["track_id"].astype(int)
mouth_df["frame_index"] = mouth_df["frame_index"].astype(int)

transcript_df["start_frame"] = transcript_df["start_frame"].astype(int)
transcript_df["end_frame"] = transcript_df["end_frame"].astype(int)

print("Mouth rows      :", len(mouth_df))
print("Track IDs       :", sorted(mouth_df["track_id"].unique().tolist()))
print("Transcript rows :", len(transcript_df))

display(mouth_df.head())
display(transcript_df.head())

Mouth rows      : 199
Track IDs       : [0, 1]
Transcript rows : 1293


,video_id,frame_index,timestamp_seconds,track_id,det_score,mouth_file,blur_score,brightness,accepted
0,video001,0,0.000000,0,0.833094,dataset\mouth_crops\video001\track_000\frame_0...,30.752,140.689,True
1,video001,0,0.000000,1,0.824791,dataset\mouth_crops\video001\track_001\frame_0...,78.174,159.318,True
2,video001,1,0.033367,0,0.834487,dataset\mouth_crops\video001\track_000\frame_0...,23.250,140.622,True
3,video001,1,0.033367,1,0.826267,dataset\mouth_crops\video001\track_001\frame_0...,68.199,160.063,True
4,video001,2,0.066733,0,0.834542,dataset\mouth_crops\video001\track_000\frame_0...,23.044,140.123,True


,video_id,segment_id,start_time,end_time,start_frame,end_frame,frame_count,text
0,video001,0,0.42,2.20,12,66,55,และนี่คือ TiaMai Podcast
1,video001,1,2.48,4.44,74,134,61,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน
2,video001,2,4.44,6.50,133,195,63,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง
3,video001,3,6.50,7.94,194,238,45,ใช่ค่ะ และทั้งหมดนะครับ
4,video001,4,7.94,9.74,237,292,56,เป็นเพลงแค่ความคิดเห็นของเราสองคน


## Cell 3 — ตรวจ Coverage

In [3]:
max_mouth_frame = int(mouth_df["frame_index"].max()) if len(mouth_df) else -1
max_transcript_frame = int(transcript_df["end_frame"].max()) if len(transcript_df) else -1

coverage = (
    max_mouth_frame / max_transcript_frame
    if max_transcript_frame > 0 else 0
)

print("Max mouth frame     :", max_mouth_frame)
print("Max transcript frame:", max_transcript_frame)
print("Approx coverage     :", f"{coverage:.2%}")

if coverage < 0.90:
    print(
        "\n[WARNING] mouth crops ยังไม่ครอบคลุมทั้งวิดีโอ "
        "แนะนำให้รัน Face Detection + Mouth Extraction ทั้งคลิปก่อน"
    )

Max mouth frame     : 99
Max transcript frame: 178258
Approx coverage     : 0.06%

[WARNING] mouth crops ยังไม่ครอบคลุมทั้งวิดีโอ แนะนำให้รัน Face Detection + Mouth Extraction ทั้งคลิปก่อน


## Cell 4 — ฟังก์ชันอ่าน Mouth + Motion

In [13]:
import cv2
import numpy as np

def resolve_mouth_path(relative_path):
    if not isinstance(relative_path, str) or not relative_path.strip():
        return None

    path = Path(relative_path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def read_mouth_gray(relative_path):
    path = resolve_mouth_path(relative_path)

    if path is None or not path.exists():
        return None

    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)

    if image is None:
        return None

    if image.shape != (96, 96):
        image = cv2.resize(image, (96, 96), interpolation=cv2.INTER_AREA)

    return image


def frame_difference(img_a, img_b):
    if img_a is None or img_b is None:
        return np.nan

    return float(cv2.absdiff(img_a, img_b).mean() / 255.0)

print("Motion functions ready")

Motion functions ready


## Cell 5 — คำนวณ Motion ต่อ Frame

In [5]:
from tqdm.auto import tqdm

motion_rows = []

for track_id, group in mouth_df.groupby("track_id"):
    group = group.sort_values("frame_index").reset_index(drop=True)

    prev_image = None
    prev_frame = None

    for _, row in tqdm(
        group.iterrows(),
        total=len(group),
        desc=f"track {track_id}"
    ):
        current_image = read_mouth_gray(row["mouth_file"])
        current_frame = int(row["frame_index"])

        if (
            prev_image is not None
            and prev_frame is not None
            and current_frame - prev_frame <= 2
        ):
            motion = frame_difference(prev_image, current_image)
        else:
            motion = np.nan

        motion_rows.append({
            "track_id": int(track_id),
            "frame_index": current_frame,
            "motion": motion,
            "blur_score": row.get("blur_score", np.nan),
            "brightness": row.get("brightness", np.nan),
        })

        prev_image = current_image
        prev_frame = current_frame

motion_df = pd.DataFrame(motion_rows)

print("Motion rows:", len(motion_df))
display(motion_df.head(20))

c:\Users\ASUS FA507NUR\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
track 1: 100%|██████████| 100/100 [00:01<00:00, 78.40it/s]

Motion rows: 199


,track_id,frame_index,motion,blur_score,brightness
0,0,0,NaN,30.752,140.689
1,0,1,0.008661,23.250,140.622
2,0,2,0.011196,23.044,140.123
3,0,3,0.004620,22.483,139.976
4,0,4,0.014087,21.376,139.716
5,0,5,0.013634,20.202,139.694
6,0,7,0.015675,20.058,139.664
7,0,8,0.011485,20.494,139.995
8,0,9,0.010747,20.970,139.565
9,0,10,0.006426,21.674,139.526


## Cell 6 — Smooth Motion

In [6]:
motion_df = motion_df.sort_values(
    ["track_id", "frame_index"]
).reset_index(drop=True)

motion_df["motion_smooth"] = (
    motion_df
    .groupby("track_id")["motion"]
    .transform(
        lambda s: s.rolling(
            window=5,
            center=True,
            min_periods=1
        ).median()
    )
)

display(motion_df.head(20))

,track_id,frame_index,motion,blur_score,brightness,motion_smooth
0,0,0,NaN,30.752,140.689,0.009928
1,0,1,0.008661,23.250,140.622,0.008661
2,0,2,0.011196,23.044,140.123,0.009928
3,0,3,0.004620,22.483,139.976,0.011196
4,0,4,0.014087,21.376,139.716,0.013634
5,0,5,0.013634,20.202,139.694,0.013634
6,0,7,0.015675,20.058,139.664,0.013634
7,0,8,0.011485,20.494,139.995,0.011485
8,0,9,0.010747,20.970,139.565,0.010747
9,0,10,0.006426,21.674,139.526,0.007801


## Cell 7 — Activity ต่อ Transcript Segment

In [7]:
segment_activity_rows = []

for _, seg in tqdm(
    transcript_df.iterrows(),
    total=len(transcript_df),
    desc="segments"
):
    segment_id = int(seg["segment_id"])
    start_frame = int(seg["start_frame"])
    end_frame = int(seg["end_frame"])

    expected_frames = max(1, end_frame - start_frame + 1)

    matched = motion_df[
        (motion_df["frame_index"] >= start_frame)
        &
        (motion_df["frame_index"] <= end_frame)
    ]

    if len(matched) == 0:
        segment_activity_rows.append({
            "segment_id": segment_id,
            "track_id": None,
            "mean_motion": np.nan,
            "median_motion": np.nan,
            "peak_motion": np.nan,
            "coverage": 0.0,
            "observed_frames": 0,
        })
        continue

    for track_id, track_data in matched.groupby("track_id"):
        valid_motion = track_data["motion_smooth"].dropna()
        observed_frames = int(track_data["frame_index"].nunique())

        segment_activity_rows.append({
            "segment_id": segment_id,
            "track_id": int(track_id),
            "mean_motion": float(valid_motion.mean()) if len(valid_motion) else np.nan,
            "median_motion": float(valid_motion.median()) if len(valid_motion) else np.nan,
            "peak_motion": float(valid_motion.quantile(0.90)) if len(valid_motion) else np.nan,
            "coverage": min(1.0, observed_frames / expected_frames),
            "observed_frames": observed_frames,
        })

activity_df = pd.DataFrame(segment_activity_rows)
display(activity_df.head(30))

segments: 100%|██████████| 1293/1293 [00:00<00:00, 1622.92it/s]


,segment_id,track_id,mean_motion,median_motion,peak_motion,coverage,observed_frames
0,0,0.0,0.014127,0.014211,0.020967,1.00000,55
1,0,1.0,0.023963,0.025917,0.031615,1.00000,55
2,1,0.0,0.013931,0.013987,0.017300,0.42623,26
3,1,1.0,0.021181,0.021704,0.025089,0.42623,26
4,2,NaN,NaN,NaN,NaN,0.00000,0
5,3,NaN,NaN,NaN,NaN,0.00000,0
6,4,NaN,NaN,NaN,NaN,0.00000,0
7,5,NaN,NaN,NaN,NaN,0.00000,0
8,6,NaN,NaN,NaN,NaN,0.00000,0
9,7,NaN,NaN,NaN,NaN,0.00000,0


## Cell 8 — Speaker Score

In [8]:
activity_df["mean_motion"] = activity_df["mean_motion"].fillna(0.0)
activity_df["peak_motion"] = activity_df["peak_motion"].fillna(0.0)
activity_df["coverage"] = activity_df["coverage"].fillna(0.0)

activity_df["speaker_score"] = (
    0.50 * activity_df["mean_motion"]
    + 0.30 * activity_df["peak_motion"]
    + 0.20 * activity_df["coverage"]
)

display(
    activity_df.sort_values(
        ["segment_id", "speaker_score"],
        ascending=[True, False]
    ).head(30)
)

,segment_id,track_id,mean_motion,median_motion,peak_motion,coverage,observed_frames,speaker_score
1,0,1.0,0.023963,0.025917,0.031615,1.00000,55,0.221466
0,0,0.0,0.014127,0.014211,0.020967,1.00000,55,0.213354
3,1,1.0,0.021181,0.021704,0.025089,0.42623,26,0.103363
2,1,0.0,0.013931,0.013987,0.017300,0.42623,26,0.097402
4,2,NaN,0.000000,NaN,0.000000,0.00000,0,0.000000
5,3,NaN,0.000000,NaN,0.000000,0.00000,0,0.000000
6,4,NaN,0.000000,NaN,0.000000,0.00000,0,0.000000
7,5,NaN,0.000000,NaN,0.000000,0.00000,0,0.000000
8,6,NaN,0.000000,NaN,0.000000,0.00000,0,0.000000
9,7,NaN,0.000000,NaN,0.000000,0.00000,0,0.000000


## Cell 9 — เลือก Candidate Speaker

In [9]:
MIN_COVERAGE = 0.50
MIN_ACTIVITY = 0.015
AUTO_MARGIN = 0.20

alignment_rows = []

for _, seg in transcript_df.iterrows():
    segment_id = int(seg["segment_id"])

    candidates = activity_df[
        activity_df["segment_id"] == segment_id
    ].copy()

    candidates = candidates[candidates["track_id"].notna()]
    candidates = candidates.sort_values("speaker_score", ascending=False)

    if len(candidates) == 0:
        alignment_rows.append({
            "video_id": VIDEO_ID,
            "segment_id": segment_id,
            "start_time": seg["start_time"],
            "end_time": seg["end_time"],
            "start_frame": int(seg["start_frame"]),
            "end_frame": int(seg["end_frame"]),
            "text": seg["text"],
            "candidate_track_id": None,
            "speaker_score": 0.0,
            "second_score": 0.0,
            "margin": 0.0,
            "coverage": 0.0,
            "status": "NO_DATA",
        })
        continue

    top = candidates.iloc[0]
    second_score = (
        float(candidates.iloc[1]["speaker_score"])
        if len(candidates) > 1 else 0.0
    )

    top_score = float(top["speaker_score"])
    margin = (top_score - second_score) / max(top_score, 1e-8)
    coverage_value = float(top["coverage"])

    if coverage_value < MIN_COVERAGE:
        status = "NO_DATA"
    elif top_score < MIN_ACTIVITY:
        status = "REVIEW"
    elif len(candidates) > 1 and margin < AUTO_MARGIN:
        status = "REVIEW"
    else:
        status = "AUTO"

    alignment_rows.append({
        "video_id": VIDEO_ID,
        "segment_id": segment_id,
        "start_time": seg["start_time"],
        "end_time": seg["end_time"],
        "start_frame": int(seg["start_frame"]),
        "end_frame": int(seg["end_frame"]),
        "text": seg["text"],
        "candidate_track_id": int(top["track_id"]),
        "speaker_score": round(top_score, 6),
        "second_score": round(second_score, 6),
        "margin": round(float(margin), 6),
        "coverage": round(coverage_value, 6),
        "status": status,
    })

alignment_df = pd.DataFrame(alignment_rows)
display(alignment_df.head(30))

,video_id,segment_id,start_time,end_time,start_frame,end_frame,text,candidate_track_id,speaker_score,second_score,margin,coverage,status
0,video001,0,0.42,2.20,12,66,และนี่คือ TiaMai Podcast,1.0,0.221466,0.213354,0.036628,1.00000,REVIEW
1,video001,1,2.48,4.44,74,134,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน,1.0,0.103363,0.097402,0.057677,0.42623,NO_DATA
2,video001,2,4.44,6.50,133,195,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง,NaN,0.000000,0.000000,0.000000,0.00000,NO_DATA
3,video001,3,6.50,7.94,194,238,ใช่ค่ะ และทั้งหมดนะครับ,NaN,0.000000,0.000000,0.000000,0.00000,NO_DATA
4,video001,4,7.94,9.74,237,292,เป็นเพลงแค่ความคิดเห็นของเราสองคน,NaN,0.000000,0.000000,0.000000,0.00000,NO_DATA
5,video001,5,9.74,11.90,291,357,ซึ่งเป็นสองคนที่ธรรมดาๆเท่านั้นเนอะ,NaN,0.000000,0.000000,0.000000,0.00000,NO_DATA
6,video001,6,11.90,15.20,356,456,ถ้าใครรู้สึกยังไงก็สามารถแสดงความคิดเห็นกันได้...,NaN,0.000000,0.000000,0.000000,0.00000,NO_DATA
7,video001,7,15.20,16.04,455,481,ใช่ค่ะ,NaN,0.000000,0.000000,0.000000,0.00000,NO_DATA
8,video001,8,20.26,22.40,607,672,และวันนี้อย่างที่ทุกคนเห็นนะคะ,NaN,0.000000,0.000000,0.000000,0.00000,NO_DATA
9,video001,9,22.40,50.76,671,1522,นี่ๆ,NaN,0.000000,0.000000,0.000000,0.00000,NO_DATA


## Cell 10 — สรุปผล

In [10]:
print("Total segments:", len(alignment_df))
print()
print(alignment_df["status"].value_counts(dropna=False))

print("\nAUTO tracks:")
display(
    alignment_df[
        alignment_df["status"] == "AUTO"
    ]["candidate_track_id"].value_counts()
)

Total segments: 1293

status
NO_DATA    1292
REVIEW        1
Name: count, dtype: int64

AUTO tracks:


Series([], Name: count, dtype: int64)

## Cell 11 — Export

In [11]:
ALL_CSV = OUTPUT_ROOT / "speaker_track_alignment.csv"
AUTO_CSV = OUTPUT_ROOT / "speaker_track_alignment_auto.csv"
REVIEW_CSV = OUTPUT_ROOT / "speaker_track_alignment_review.csv"
MOTION_CSV = OUTPUT_ROOT / "mouth_motion.csv"
ACTIVITY_CSV = OUTPUT_ROOT / "segment_track_activity.csv"

alignment_df.to_csv(ALL_CSV, index=False, encoding="utf-8-sig")

alignment_df[
    alignment_df["status"] == "AUTO"
].to_csv(
    AUTO_CSV,
    index=False,
    encoding="utf-8-sig"
)

alignment_df[
    alignment_df["status"] != "AUTO"
].to_csv(
    REVIEW_CSV,
    index=False,
    encoding="utf-8-sig"
)

motion_df.to_csv(MOTION_CSV, index=False, encoding="utf-8-sig")
activity_df.to_csv(ACTIVITY_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print(" -", ALL_CSV)
print(" -", AUTO_CSV)
print(" -", REVIEW_CSV)
print(" -", MOTION_CSV)
print(" -", ACTIVITY_CSV)

Saved:
 - D:\Face_detection\dataset\speaker_alignment\video001\speaker_track_alignment.csv
 - D:\Face_detection\dataset\speaker_alignment\video001\speaker_track_alignment_auto.csv
 - D:\Face_detection\dataset\speaker_alignment\video001\speaker_track_alignment_review.csv
 - D:\Face_detection\dataset\speaker_alignment\video001\mouth_motion.csv
 - D:\Face_detection\dataset\speaker_alignment\video001\segment_track_activity.csv


## Cell 12 — Preview REVIEW

In [12]:
review_df = alignment_df[
    alignment_df["status"] != "AUTO"
].copy()

display(
    review_df[
        [
            "segment_id",
            "start_time",
            "end_time",
            "text",
            "candidate_track_id",
            "speaker_score",
            "margin",
            "coverage",
            "status",
        ]
    ].head(50)
)

,segment_id,start_time,end_time,text,candidate_track_id,speaker_score,margin,coverage,status
0,0,0.42,2.20,และนี่คือ TiaMai Podcast,1.0,0.221466,0.036628,1.00000,REVIEW
1,1,2.48,4.44,Podcast ที่จะชวนคุณมาวิคราบไปด้วยกัน,1.0,0.103363,0.057677,0.42623,NO_DATA
2,2,4.44,6.50,ว่าถ้ามันเกิดขึ้นกับคุณ คุณจะทำยังไง,NaN,0.000000,0.000000,0.00000,NO_DATA
3,3,6.50,7.94,ใช่ค่ะ และทั้งหมดนะครับ,NaN,0.000000,0.000000,0.00000,NO_DATA
4,4,7.94,9.74,เป็นเพลงแค่ความคิดเห็นของเราสองคน,NaN,0.000000,0.000000,0.00000,NO_DATA
5,5,9.74,11.90,ซึ่งเป็นสองคนที่ธรรมดาๆเท่านั้นเนอะ,NaN,0.000000,0.000000,0.00000,NO_DATA
6,6,11.90,15.20,ถ้าใครรู้สึกยังไงก็สามารถแสดงความคิดเห็นกันได้...,NaN,0.000000,0.000000,0.00000,NO_DATA
7,7,15.20,16.04,ใช่ค่ะ,NaN,0.000000,0.000000,0.00000,NO_DATA
8,8,20.26,22.40,และวันนี้อย่างที่ทุกคนเห็นนะคะ,NaN,0.000000,0.000000,0.00000,NO_DATA
9,9,22.40,50.76,นี่ๆ,NaN,0.000000,0.000000,0.00000,NO_DATA


# ขั้นต่อไป — 05 Labeled Lip Clip Builder

เมื่อ alignment ดีพอ เราจะ:
1. ใช้เฉพาะ `AUTO` ก่อน
2. ดึง mouth frames ตาม start/end frame
3. reject segment ที่เฟรมขาดเยอะ
4. สร้าง clip และ manifest
5. เตรียม train / val / test